<div style="background:linear-gradient(120deg,#1A73E8 0%,#6A3DE8 50%,#D93025 100%);padding:30px 32px;border-radius:16px;color:#fff">
  <div style="font-size:13px;letter-spacing:2px;opacity:.85;color:#fff">GOOGLE · THE GEMMA 4 DEVELOPER AGENT COMPETITION</div>
  <h1 style="color:#fff;margin:6px 0 4px 0;font-size:34px">🤖 Universal Starter: EDA → Localization → ADK Agent → <code style="background:rgba(255,255,255,.18);color:#fff;padding:2px 8px;border-radius:6px">submission.zip</code></h1>
  <p style="font-size:16px;margin:10px 0 0 0;color:#fff;opacity:.95">One config cell · Run-All safe · every harness fact verified against the competition files at runtime</p>
</div>

<br>

**The task in one sentence:** build a software-engineering agent on **Gemma 4** that reads a real GitHub issue, edits a Python repository inside a sandbox, and produces a patch that makes hidden tests pass. Submissions are **declarative agent bundles** (YAML + prompts, optionally adapters) rather than arbitrary code.

### ✨ What makes this starter different

| | Principle | Why it matters |
|:-:|:--|:--|
| 🔎 | **Zero hard-coded harness facts** | Tool names, model alias, context limit and bundle schema are *detected* from `HARNESS_README.md` and `sample_submission`, with the evidence shown. If the organizers change something, the notebook adapts. |
| 🛡️ | **Run-All safe** | Every section degrades gracefully. Unknown data formats produce a warning, not a crash. |
| 📐 | **Correct EDA** | A real unified-diff parser (hunk-count aware, works with or without `diff --git` headers). All takeaways are *computed*, not guessed. |
| 🎯 | **Localization baseline** | A no-LLM TF-IDF retriever measures how hard it is to find the file to fix. That is the #1 failure mode of SWE agents. |
| 🧰 | **Reusable tooling** | `edit_file` simulator with syntax checks, patch-sanity checker, tag-tolerant YAML validator, deterministic zip packaging. |
| 🚫 | **No leakage** | Reference patches are only used for analysis and are never written to any output. |

### 🗺️ Contents
1. [⚙️ Config](#config) : the only cell you need to edit
2. [📦 Data discovery](#discovery) : locate the data, inventory it, read the harness README
3. [🧰 Tool verification](#tools) : which agent tools really exist
4. [📊 Task EDA](#eda) : repositories, patch anatomy, difficulty, timeline
5. [🧭 Localization signals](#signals) : what the issue text tells us about *where* to edit
6. [🕸️ Code intelligence](#graphs) : call graphs & embeddings explorer
7. [🎯 Localization baseline](#baseline) : lexical retrieval Hit@k
8. [✂️ Edit mechanics & patch safety](#edit) : avoid the classic `edit_file` pitfalls
9. [🧠 Prompts & context budget](#prompts)
10. [🏗️ Agent bundle builder](#bundle) : single agent or analyzer + coder
11. [🛡️ Validator & packaging](#validate) : `submission.zip`
12. [🚀 Playbook](#playbook) : what to try next, ranked

> **Quick start:** attach the competition data → (optionally) edit `CFG` → **Run All** → download `submission.zip` from the output panel and submit it following the competition's submission instructions.

<a id="config"></a>
# ⚙️ 1. Config
Everything tunable lives here. `None` means *auto-detect from the competition files*.

In [ ]:
CFG = {
    # ── agent ──────────────────────────────────────────────────────────────
    "architecture": "analyzer+coder",   # "single" | "analyzer+coder"
    "bundle_base": "starter",           # "starter" = build from this notebook's templates
                                        # "sample"  = copy sample_submission and only swap in our prompt
    "model_alias": None,                # None → detected from sample_submission
    "fallback_model_alias": "gemma-4-31b-it-qat-w4a16-ct",
    "temperature": 0.2,
    "top_p": 0.95,
    "top_k": 40,
    "max_output_tokens": 8192,
    "thinking_budget": 4096,
    # ── environment ────────────────────────────────────────────────────────
    "context_limit": None,              # None → detected from HARNESS_README.md
    "fallback_context_limit": 32768,
    # ── analysis ───────────────────────────────────────────────────────────
    "localization_max_tasks": 200,
    "graph_scan_max_tasks": 60,
    "time_budget_s": 240,               # soft cap per heavy analysis loop
    "seed": 42,
    # ── output ─────────────────────────────────────────────────────────────
    "bundle_dir": "submission_bundle",
    "zip_name": "submission.zip",
}

<a id="discovery"></a>
# 📦 2. Setup & data discovery

In [ ]:
import os, re, sys, json, gzip, time, html, pickle, shutil, tarfile, zipfile, hashlib, textwrap, warnings
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import yaml
from IPython.display import display, HTML, Markdown

from pandas.io.formats.style import Styler
if not hasattr(Styler, "map"):          # pandas < 2.1 compatibility
    Styler.map = Styler.applymap

warnings.filterwarnings("ignore")
np.random.seed(CFG["seed"])
pd.set_option("display.max_colwidth", 90)

# ── visual identity ─────────────────────────────────────────────────────────
PAL = ["#4285F4", "#EA4335", "#FBBC04", "#34A853", "#A142F4", "#24C1E0", "#F538A0", "#5F6368"]
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({
    "figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans",
    "figure.titleweight": "bold", "figure.titlesize": 15,
})

def esc(s):
    return html.escape(str(s))

def callout(msg, kind="info"):
    bg, bd, icon = {"info": ("#E8F0FE", "#4285F4", "💡"), "ok": ("#E6F4EA", "#34A853", "✅"),
                    "warn": ("#FEF7E0", "#F9AB00", "⚠️"), "err": ("#FCE8E6", "#EA4335", "⛔")}[kind]
    display(HTML(f'<div style="background:{bg};border-left:5px solid {bd};padding:10px 14px;'
                 f'border-radius:6px;margin:6px 0;color:#202124">{icon} {msg}</div>'))

def kpis(items):
    cards = "".join(
        f'<div style="flex:1;min-width:130px;background:#fff;border:1px solid #DADCE0;border-radius:12px;'
        f'padding:10px 14px;margin:4px;box-shadow:0 1px 2px rgba(60,64,67,.15)">'
        f'<div style="font-size:12px;color:#5F6368">{esc(label)}</div>'
        f'<div style="font-size:22px;font-weight:700;color:#1A73E8">{esc(value)}</div></div>'
        for label, value in items)
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;margin:4px 0 10px 0">{cards}</div>'))

def details(summary, body_html):
    display(HTML(f'<details style="margin:4px 0"><summary style="cursor:pointer;font-weight:600">{esc(summary)}</summary>{body_html}</details>'))

def pre(text, max_lines=80):
    lines = str(text).splitlines()
    more = f"\n… ({len(lines) - max_lines} more lines)" if len(lines) > max_lines else ""
    return ('<pre style="font-size:12px;background:#F8F9FA;color:#202124;padding:10px;border-radius:8px;overflow-x:auto">'
            + esc("\n".join(lines[:max_lines]) + more) + "</pre>")

def human_bytes(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:,.1f} {unit}" if unit != "B" else f"{n} B"
        n /= 1024
    return f"{n:.1f} PB"

@contextmanager
def safe(name):
    """Run a section; on failure show a warning instead of crashing Run All."""
    try:
        yield
    except Exception as e:
        callout(f"<b>{esc(name)}</b> skipped: <code>{esc(type(e).__name__)}: {esc(e)}</code>", "warn")

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · networkx {nx.__version__} · working dir: {WORK}")

### 🔎 Locating the competition data
No hard-coded personal paths: we check the standard Kaggle mount points, then search `/kaggle/input` for `tasks.jsonl`. You can also set `GEMMA_AGENT_DATA=/path/to/data` to run this notebook locally.

In [ ]:
def limited_walk(root, max_depth):
    root = Path(root)
    if not root.exists():
        return
    stack = [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        for e in entries:
            yield Path(e.path)
            if e.is_dir(follow_symlinks=False) and depth < max_depth:
                stack.append((Path(e.path), depth + 1))

def find_data_dir():
    candidates = [Path(os.environ["GEMMA_AGENT_DATA"])] if os.environ.get("GEMMA_AGENT_DATA") else []
    candidates += [Path("/kaggle/input/competitions/gemma-4-developer-agent"),
                   Path("/kaggle/input/gemma-4-developer-agent"),
                   Path("competition_data"), Path("../competition_data")]
    for c in candidates:
        if (c / "tasks.jsonl").exists():
            return c
    for root in [Path("/kaggle/input"), Path(".")]:
        for p in limited_walk(root, 3):
            if p.name == "tasks.jsonl":
                return p.parent
    return None

DATA_DIR = find_data_dir()
DEMO_MODE = DATA_DIR is None
if DEMO_MODE:
    callout("Competition data not found → running in <b>demo mode</b> on a tiny synthetic sample. "
            "Attach the competition dataset (<i>Add Input → Competitions</i>) for real results.", "warn")
else:
    callout(f"Data directory: <code>{esc(DATA_DIR)}</code>", "ok")

In [ ]:
def dir_stats(path, max_files=200_000):
    n, size, exts, truncated = 0, 0, Counter(), False
    for dp, _, fns in os.walk(path):
        for f in fns:
            n += 1
            try:
                size += os.path.getsize(os.path.join(dp, f))
            except OSError:
                pass
            name = f.lower()
            ext = ".tar.gz" if name.endswith(".tar.gz") else (Path(f).suffix.lower() or "(none)")
            exts[ext] += 1
            if n >= max_files:
                truncated = True
                break
        if truncated:
            break
    return n, size, exts, truncated

INVENTORY = pd.DataFrame()
if not DEMO_MODE:
    rows = []
    for e in sorted(DATA_DIR.iterdir()):
        if e.is_dir():
            n, s, exts, tr = dir_stats(e)
            kind = "📁"
        else:
            n, s, exts, tr = 1, e.stat().st_size, Counter({e.suffix or "(none)": 1}), False
            kind = "📄"
        rows.append({"": kind, "entry": e.name, "files": f"{n:,}{'+' if tr else ''}", "size": human_bytes(s),
                     "top extensions": ", ".join(f"{k} ×{v}" for k, v in exts.most_common(4))})
    INVENTORY = pd.DataFrame(rows)
    display(INVENTORY.style.hide(axis="index").set_properties(**{"text-align": "left"}))

### 📜 The harness README is the source of truth
We extract key facts **with the line they came from**, so you can check every claim at a glance. The full README is rendered below (collapsed).

In [ ]:
README_PATH = None
if not DEMO_MODE:
    README_PATH = next((p for p in [DATA_DIR / "HARNESS_README.md", *sorted(DATA_DIR.glob("*README*"))] if p.exists()), None)
README = README_PATH.read_text(encoding="utf-8", errors="replace") if README_PATH else ""

def readme_lines(pattern, max_hits=3):
    hits = []
    for line in README.splitlines():
        if re.search(pattern, line, flags=re.I) and line.strip():
            hits.append(textwrap.shorten(line.strip(), 170))
            if len(hits) >= max_hits:
                break
    return hits

def detect_context_limit(text):
    """Look for max_model_len / context-window numbers; return (value, evidence line)."""
    pats = [r"max[_ -]?model[_ -]?len\D{0,20}?(\d{1,3}(?:[,_]\d{3})+|\d{4,7})",
            r"context(?:[ -]window|[ -]length|[ -]limit|[ -]ceiling)?\D{0,40}?(\d{1,3}(?:[,_]\d{3})+|\d{4,7})\s*(?:tokens)?"]
    for pat in pats:
        for line in text.splitlines():
            m = re.search(pat, line, flags=re.I)
            if m:
                v = int(re.sub(r"[,_]", "", m.group(1)))
                if 2048 <= v <= 2_000_000:
                    return v, line.strip()
    return None, None

ctx_value, ctx_evidence = detect_context_limit(README)
CONTEXT_LIMIT = CFG["context_limit"] or ctx_value or CFG["fallback_context_limit"]
ctx_source = "CFG override" if CFG["context_limit"] else ("HARNESS_README.md" if ctx_value else "fallback (not found in README)")

facts = []
for topic, pat in [("Context / tokens", r"max_model_len|context|token"), ("Turn / call budget", r"turn|budget|max[_ ]?calls?"),
                   ("Time limits", r"timeout|wall|second|minute"), ("Hardware", r"\bGPU|L4|A100|H100|TPU|vLLM"),
                   ("Submission rules", r"submission|archive|\.zip|extension|adapter|LoRA"),
                   ("Scoring", r"score|metric|pass|resolved|pytest")]:
    for line in readme_lines(pat):
        facts.append({"topic": topic, "evidence (README line)": line})

if README:
    callout(f"Context limit used by this notebook: <b>{CONTEXT_LIMIT:,} tokens</b> (source: {esc(ctx_source)})"
            + (f"<br><small>evidence: <code>{esc(ctx_evidence)}</code></small>" if ctx_evidence else ""), "info")
    if facts:
        display(pd.DataFrame(facts).style.hide(axis="index").set_properties(**{"text-align": "left"}))
    details(f"📖 Show full {README_PATH.name} ({len(README.splitlines())} lines)", pre(README, max_lines=400))
else:
    callout(f"No harness README found. Using fallback context limit {CONTEXT_LIMIT:,}.", "warn")

<a id="tools"></a>
# 🧰 3. Tool verification
Community notebooks list up to nine sandbox tools. Rather than trusting any list (including this one), we **grep the harness files** for each tool name. The agent bundle built later only uses tools that were actually found, so a typo or a renamed tool cannot silently break your submission.

In [ ]:
EXPECTED_TOOLS = {
    "run_command": "execution", "submit_patch": "execution", "get_status": "execution",
    "read_file": "filesystem", "edit_file": "filesystem", "write_file": "filesystem",
    "search_similar_code": "code intelligence", "get_code_neighbors": "code intelligence",
    "get_code_subgraph": "code intelligence",
}
TEXT_EXTS = {".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml", ".cfg", ".sh", ".ini"}

def text_corpus(dirs, max_file_bytes=2_000_000, max_files=4000):
    chunks, n = [], 0
    for d in dirs:
        if not d or not Path(d).exists():
            continue
        for p in limited_walk(d, 6):
            if p.is_file() and p.suffix.lower() in TEXT_EXTS:
                try:
                    if p.stat().st_size <= max_file_bytes:
                        chunks.append(p.read_text(encoding="utf-8", errors="ignore"))
                        n += 1
                except OSError:
                    pass
            if n >= max_files:
                break
    return "\n".join(chunks)

HARNESS_DIRS = [] if DEMO_MODE else [DATA_DIR / d for d in ("sandbox", "docker")]
SAMPLE_DIRS = [] if DEMO_MODE else [p for p in DATA_DIR.iterdir() if "sample" in p.name.lower() and p.is_dir()]
corpus_harness = text_corpus(HARNESS_DIRS)
corpus_sample = text_corpus(SAMPLE_DIRS)

rows = []
for tool, cat in EXPECTED_TOOLS.items():
    pat = re.compile(r"\b" + re.escape(tool) + r"\b")
    r, h, s = bool(pat.search(README)), bool(pat.search(corpus_harness)), bool(pat.search(corpus_sample))
    rows.append({"tool": tool, "category": cat, "README": "✅" if r else "—", "harness sources": "✅" if h else "—",
                 "sample submission": "✅" if s else "—", "verified": r or h or s})
TOOLS_DF = pd.DataFrame(rows)

have_evidence = bool(README or corpus_harness or corpus_sample)
AVAILABLE_TOOLS = [t for t in EXPECTED_TOOLS if TOOLS_DF.set_index("tool").loc[t, "verified"]] if have_evidence else list(EXPECTED_TOOLS)

def _style_verified(v):
    return "background-color:#E6F4EA;color:#137333;font-weight:600" if v is True else (
           "background-color:#FCE8E6;color:#C5221F;font-weight:600" if v is False else "")
display(TOOLS_DF.style.hide(axis="index").map(_style_verified, subset=["verified"]))

other = sorted(set(re.findall(r"`([a-z_][a-z0-9_]{2,})\(", README)) - set(EXPECTED_TOOLS))
if other:
    callout("Other callables mentioned in the README (review, some may be tools): "
            + ", ".join(f"<code>{esc(o)}</code>" for o in other), "info")
if not have_evidence:
    callout("No harness files to verify against → assuming the default tool list.", "warn")
missing = [t for t in EXPECTED_TOOLS if t not in AVAILABLE_TOOLS]
if missing and have_evidence:
    callout("Not found anywhere, excluded from the bundle: " + ", ".join(f"<code>{m}</code>" for m in missing), "warn")
print("AVAILABLE_TOOLS =", AVAILABLE_TOOLS)

<a id="eda"></a>
# 📊 4. Task EDA
### Loading `tasks.jsonl` and its schema

In [ ]:
DEMO_TASKS = [
    {"instance_id": "demo_1", "repo": "demo/pkg", "base_commit": "0" * 40,
     "problem_statement": "`parse_query` crashes on empty input.\n\nTraceback (most recent call last):\nValueError",
     "hints_text": "", "created_at": "2026-05-01T00:00:00Z",
     "patch": "--- a/pkg/query.py\n+++ b/pkg/query.py\n@@ -1,3 +1,5 @@ def parse_query(s):\n     x = 1\n+    if not s:\n+        return {}\n     y = 2\n     z = 3\n",
     "test_patch": "--- a/tests/test_query.py\n+++ b/tests/test_query.py\n@@ -1,1 +1,3 @@\n import pkg\n+def test_empty():\n+    assert pkg.parse_query('') == {}\n"},
    {"instance_id": "demo_2", "repo": "demo/pkg", "base_commit": "1" * 40,
     "problem_statement": "Headers with underscores are silently accepted.", "hints_text": "see headers.py",
     "created_at": "2026-06-01T00:00:00Z",
     "patch": "diff --git a/pkg/headers.py b/pkg/headers.py\n--- a/pkg/headers.py\n+++ b/pkg/headers.py\n@@ -10,2 +10,2 @@ class Headers:\n-    allow = True\n+    allow = False\n     pass\n",
     "test_patch": ""},
]

if not DEMO_MODE:
    df = pd.read_json(DATA_DIR / "tasks.jsonl", lines=True, dtype=False)
else:
    df = pd.DataFrame(DEMO_TASKS)

# normalise the columns this notebook relies on
if "instance_id" not in df.columns:
    df["instance_id"] = df["id"] if "id" in df.columns else [f"task_{i}" for i in range(len(df))]
for col in ["problem_statement", "hints_text", "patch", "test_patch", "base_commit"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)
if "repo" not in df.columns:
    df["repo"] = df["instance_id"].str.rsplit("_", n=1).str[0]
df["created_at"] = pd.to_datetime(df.get("created_at"), errors="coerce", utc=True)

def schema_table(frame):
    rows = []
    for c in frame.columns:
        s = frame[c]
        filled = s.notna() & s.astype(str).str.strip().ne("") & s.astype(str).ne("NaT")
        example = next((v for v in s if isinstance(v, str) and v.strip()), s.iloc[0] if len(s) else "")
        rows.append({"column": c, "dtype": str(s.dtype), "filled %": round(100 * filled.mean(), 1),
                     "unique": s.astype(str).nunique(),
                     "example": textwrap.shorten(str(example).replace("\n", " ⏎ "), 80)})
    return pd.DataFrame(rows)

print(f"{len(df)} tasks · {df['repo'].nunique()} repositories · {len(df.columns)} columns")
display(schema_table(df).style.hide(axis="index")
        .bar(subset=["filled %"], color="#AECBFA", vmin=0, vmax=100).format({"filled %": "{:.0f}%"})
        .set_properties(**{"text-align": "left"}))

### 🔬 A real unified-diff parser
The reference patches use plain `--- a/` / `+++ b/` headers, **often without** a `diff --git` line. Counting `diff --git` lines therefore reports "1 file" for every task. The parser below tracks hunk line counts, so it handles both styles and also catches malformed patches. That last part is useful later for checking your agent's output.

In [ ]:
HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@(.*)$")
SYMBOL_RE = re.compile(r"(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)")

def _clean_path(p):
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p

def parse_patch(text):
    """Return {'files': [...], 'malformed': int, 'unterminated': bool}. Each file: path, added, deleted, hunks, symbols, status."""
    files, cur, old_path = [], None, None
    rem_old = rem_new = 0
    malformed = 0

    def new_file(path):
        f = {"path": path, "added": 0, "deleted": 0, "hunks": 0, "symbols": set(), "status": "modified", "_hdr": False}
        files.append(f)
        return f

    for line in (text or "").splitlines():
        if rem_old > 0 or rem_new > 0:                      # inside a hunk
            tag = line[:1]
            if tag == "+":
                cur["added"] += 1; rem_new -= 1
                m = SYMBOL_RE.match(line[1:].strip());  m and cur["symbols"].add(m.group(1))
                continue
            if tag == "-":
                cur["deleted"] += 1; rem_old -= 1
                m = SYMBOL_RE.match(line[1:].strip());  m and cur["symbols"].add(m.group(1))
                continue
            if tag == " " or line == "":
                rem_old -= 1; rem_new -= 1
                continue
            if tag == "\\":
                continue
            malformed += 1
            rem_old = rem_new = 0                            # fall through: treat as header
        if line.startswith("\\"):
            continue
        if line.startswith("diff --git "):
            m = re.match(r"diff --git a/(\S+) b/(\S+)", line)
            cur = new_file(m.group(2) if m else line[11:].strip())
            continue
        if line.startswith("--- "):
            old_path = _clean_path(line[4:])
            continue
        if line.startswith("+++ "):
            new_path = _clean_path(line[4:])
            if cur is None or cur["_hdr"]:
                cur = new_file(new_path)
            cur["_hdr"] = True
            if new_path == "/dev/null":
                cur["path"], cur["status"] = old_path, "deleted"
            else:
                cur["path"] = new_path
                if old_path == "/dev/null":
                    cur["status"] = "added"
            continue
        m = HUNK_RE.match(line)
        if m and cur is not None:
            rem_old = int(m.group(2)) if m.group(2) is not None else 1
            rem_new = int(m.group(4)) if m.group(4) is not None else 1
            cur["hunks"] += 1
            sm = SYMBOL_RE.search(m.group(5) or "")
            if sm:
                cur["symbols"].add(sm.group(1))
    for f in files:
        f.pop("_hdr", None)
        f["symbols"] = sorted(f["symbols"])
    return {"files": files, "malformed": malformed, "unterminated": rem_old > 0 or rem_new > 0}

def is_test_path(p):
    p = p.lower()
    return bool(re.search(r"(^|/)(tests?|testing)/", p) or re.search(r"(^|/)test_[^/]*\.py$", p) or p.endswith("_test.py") or p.endswith("conftest.py"))

# ── self-test: both header styles, symbols, hunk accounting ───────────────────────
_t1 = parse_patch("--- a/x.py\n+++ b/x.py\n@@ -1,2 +1,3 @@ def foo():\n a\n+b\n c\n--- a/y.py\n+++ b/y.py\n@@ -5 +5 @@\n-q\n+r\n")
_t2 = parse_patch("diff --git a/n.py b/n.py\nnew file mode 100644\n--- /dev/null\n+++ b/n.py\n@@ -0,0 +1,2 @@\n+class Bar:\n+    pass\n")
assert [f["path"] for f in _t1["files"]] == ["x.py", "y.py"] and _t1["files"][0]["symbols"] == ["foo"]
assert _t1["files"][1]["added"] == 1 and not _t1["unterminated"] and _t1["malformed"] == 0
assert _t2["files"][0]["status"] == "added" and _t2["files"][0]["symbols"] == ["Bar"]
print("parse_patch self-test passed ✅")

In [ ]:
def task_features(row):
    p = parse_patch(row["patch"])
    src = [f for f in p["files"] if not is_test_path(f["path"])]
    tp = parse_patch(row["test_patch"])
    new_tests = re.findall(r"^\+\s*(?:async\s+)?def\s+(test_\w+)", row["test_patch"], flags=re.M)
    added = sum(f["added"] for f in p["files"]); deleted = sum(f["deleted"] for f in p["files"])
    return pd.Series({
        "gold_files": [f["path"] for f in src],
        "gold_symbols": sorted({s for f in src for s in f["symbols"]}),
        "n_files": len(p["files"]), "n_src_files": len(src),
        "n_hunks": sum(f["hunks"] for f in p["files"]),
        "added": added, "deleted": deleted, "churn": added + deleted,
        "adds_new_file": any(f["status"] == "added" for f in p["files"]),
        "n_test_files": len(tp["files"]), "n_new_tests": len(new_tests),
        "patch_ok": (not p["malformed"]) and (not p["unterminated"]) and bool(p["files"]),
    })

feat = df.apply(task_features, axis=1)
df = pd.concat([df.drop(columns=[c for c in feat.columns if c in df.columns]), feat], axis=1)

df["problem_words"] = df["problem_statement"].str.split().str.len().fillna(0).astype(int)
df["has_hints"] = df["hints_text"].str.strip().ne("")
BINS, LABELS = [-1, 5, 20, 80, 300, np.inf], ["XS (≤5)", "S (≤20)", "M (≤80)", "L (≤300)", "XL (>300)"]
df["size_bucket"] = pd.cut(df["churn"], bins=BINS, labels=LABELS)

for col in ["FAIL_TO_PASS", "PASS_TO_PASS"]:           # present in some SWE-bench style dumps
    if col in df.columns:
        df[f"n_{col.lower()}"] = df[col].apply(lambda v: len(json.loads(v)) if isinstance(v, str) and v.startswith("[") else (len(v) if isinstance(v, list) else np.nan))

span = (f"{df['created_at'].min():%Y-%m} → {df['created_at'].max():%Y-%m}" if df["created_at"].notna().any() else "n/a")
kpis([("Tasks", f"{len(df):,}"), ("Repositories", df["repo"].nunique()),
      ("Median churn (lines)", int(df["churn"].median())), ("Single-file fixes", f"{(df['n_src_files'] == 1).mean():.0%}"),
      ("With hints", f"{df['has_hints'].mean():.0%}"), ("Adds new tests", f"{(df['n_new_tests'] > 0).mean():.0%}"),
      ("Well-formed patches", f"{df['patch_ok'].mean():.0%}"), ("Created", span)])

### 🎨 The benchmark at a glance

In [ ]:
with safe("EDA dashboard"):
    repo_order = df["repo"].value_counts().index.tolist()
    rc = {r: PAL[i % len(PAL)] for i, r in enumerate(repo_order)}
    fig, ax = plt.subplots(2, 3, figsize=(18, 9.5))
    fig.suptitle("Gemma 4 Developer Agent: benchmark anatomy", y=1.0)

    # (a) tasks per repo
    vc = df["repo"].value_counts()
    a = ax[0, 0]; a.barh(vc.index[::-1], vc.values[::-1], color=[rc[r] for r in vc.index[::-1]])
    for i, v in enumerate(vc.values[::-1]):
        a.text(v, i, f" {v}", va="center", fontsize=9)
    a.set_title("Tasks per repository"); a.set_xlabel("tasks")

    # (b) difficulty buckets by repo
    a = ax[0, 1]
    ct = pd.crosstab(df["repo"], df["size_bucket"]).reindex(index=repo_order[::-1], columns=LABELS, fill_value=0)
    left = np.zeros(len(ct))
    for j, lab in enumerate(LABELS):
        a.barh(ct.index, ct[lab], left=left, color=sns.color_palette("viridis", len(LABELS))[j], label=lab)
        left += ct[lab].values
    a.set_title("Patch size buckets (churn = added + deleted)"); a.legend(fontsize=8, ncol=2, loc="lower right")

    # (c) churn distribution (log)
    a = ax[0, 2]
    bins = np.logspace(0, np.log10(max(2, df["churn"].max() + 1)), 30)
    a.hist(df["churn"].clip(lower=1), bins=bins, color=PAL[0], alpha=.85, edgecolor="white")
    a.set_xscale("log"); a.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    a.axvline(df["churn"].median(), color=PAL[1], ls="--", lw=2, label=f"median = {int(df['churn'].median())}")
    a.set_title("Reference patch churn (log scale)"); a.set_xlabel("lines changed"); a.legend()

    # (d) files & hunks
    a = ax[1, 0]
    fcount = df["n_src_files"].clip(upper=5).value_counts().sort_index()
    hcount = df["n_hunks"].clip(upper=8).value_counts().sort_index()
    x = np.arange(1, 9)
    a.bar(x - .2, [fcount.get(i, 0) for i in x], width=.4, label="source files", color=PAL[3])
    a.bar(x + .2, [hcount.get(i, 0) for i in x], width=.4, label="hunks", color=PAL[4])
    a.set_xticks(x, [str(i) if i < 8 else "8+" for i in x]); a.set_title("How spread out is a fix?")
    a.set_xlabel("count per task (files capped at 5+, hunks at 8+)"); a.legend()

    # (e) problem statement length by repo
    a = ax[1, 1]
    sns.boxplot(data=df.assign(problem_words=df["problem_words"].clip(lower=1)), y="repo", x="problem_words",
                hue="repo", order=repo_order, hue_order=repo_order, palette=rc, legend=False, ax=a, fliersize=2)
    a.set_xscale("log"); a.set_title("Issue length (words, log)"); a.set_ylabel(""); a.set_xlabel("words")
    a.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    a.xaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())

    # (f) timeline
    a = ax[1, 2]
    if df["created_at"].notna().any():
        tl = (df.dropna(subset=["created_at"]).assign(month=lambda d: d["created_at"].dt.strftime("%Y-%m"))
                .pivot_table(index="month", columns="repo", values="instance_id", aggfunc="count", fill_value=0))
        tl = tl.reindex(columns=[r for r in repo_order if r in tl.columns])
        tl.plot(kind="bar", stacked=True, ax=a, color=[rc[r] for r in tl.columns], width=.85, legend=False)
        a.set_title("When were the tasks created?"); a.set_xlabel(""); a.tick_params(axis="x", rotation=45)
    else:
        a.text(.5, .5, "no created_at column", ha="center"); a.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=rc[r]) for r in repo_order]
    fig.legend(handles, repo_order, loc="lower center", ncol=min(6, len(repo_order)), bbox_to_anchor=(.5, -.03), frameon=False)
    plt.tight_layout(); plt.show()

In [ ]:
with safe("EDA takeaways"):
    single = (df["n_src_files"] == 1).mean(); small = (df["churn"] <= 20).mean()
    q90 = int(df["churn"].quantile(.9)); multi_hunk = (df["n_hunks"] > 1).mean()
    top_repo = df["repo"].value_counts()
    txt = f"""
#### 📌 Computed takeaways
- **{single:.0%}** of fixes touch exactly one source file, and **{multi_hunk:.0%}** need more than one hunk. So the agent must be able to make *several coordinated edits*, not just one.
- **{small:.0%}** of reference patches change ≤ 20 lines, while the 90th percentile is **{q90} lines**. There is a long tail of larger feature-style changes.
- **{(df['n_new_tests'] > 0).mean():.0%}** of tasks add new test functions. Those hidden tests check behaviour the issue describes, so reproduce it yourself before you fix it.
- The biggest repository (`{top_repo.index[0]}`) accounts for **{top_repo.iloc[0] / len(df):.0%}** of tasks. Repo-specific prompt hints (layout, test commands) can pay off.
- **{df['adds_new_file'].mean():.0%}** of fixes create a new file, so `write_file` matters, but rarely.
"""
    display(Markdown(txt))

<a id="signals"></a>
# 🧭 5. Localization signals
An agent that edits the wrong file cannot pass. How often does the issue text itself *point at* the fix? We check, per task, whether the problem statement mentions a gold file (path or module name), a gold symbol (function/class changed by the reference patch), contains a traceback, or includes a code block.

In [ ]:
GENERIC_STEMS = {"__init__", "utils", "util", "main", "base", "core", "types", "models", "helpers", "common", "compat"}

def mentions(text, needles):
    t = text.lower()
    return any(re.search(r"(?<![\w/])" + re.escape(n.lower()) + r"(?![\w])", t) for n in needles if len(n) >= 3)

def signal_row(r):
    text = r["problem_statement"] + "\n" + r["hints_text"]
    paths = r["gold_files"]
    stems = [Path(p).stem for p in paths if Path(p).stem not in GENERIC_STEMS and len(Path(p).stem) >= 4]
    return pd.Series({
        "mentions_file_path": mentions(text, paths),
        "mentions_module": mentions(text, stems),
        "mentions_symbol": mentions(text, r["gold_symbols"]),
        "has_traceback": "Traceback (most recent call last)" in text,
        "has_code_block": "```" in text or "\n    " in r["problem_statement"],
        "has_hints": r["has_hints"],
    })

SIGNALS = df.apply(signal_row, axis=1)
df = pd.concat([df.drop(columns=[c for c in SIGNALS.columns if c in df.columns and c != "has_hints"]),
                SIGNALS.drop(columns=["has_hints"])], axis=1)
df["any_location_hint"] = df[["mentions_file_path", "mentions_module", "mentions_symbol", "has_traceback"]].any(axis=1)

with safe("signals chart"):
    sig_cols = ["mentions_file_path", "mentions_module", "mentions_symbol", "has_traceback", "has_code_block", "has_hints", "any_location_hint"]
    by_repo = df.groupby("repo")[sig_cols].mean().T * 100
    by_repo["ALL"] = df[sig_cols].mean() * 100
    fig, ax = plt.subplots(figsize=(11, 4.6))
    sns.heatmap(by_repo, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of tasks"}, ax=ax, vmin=0, vmax=100)
    ax.set_title("What does the issue text reveal about the fix location?  (% of tasks)")
    ax.set_ylabel(""); ax.set_xlabel("")
    plt.tight_layout(); plt.show()
    no_hint = 1 - df["any_location_hint"].mean()
    callout(f"<b>{no_hint:.0%}</b> of tasks give <i>no direct location hint</i> (no file, module, symbol or traceback). "
            "For these, the agent has to search. That is where graph and embedding tools or a grep-first strategy earn their keep.", "info")

In [ ]:
with safe("hot files"):
    hot = (df[["repo", "gold_files"]].explode("gold_files").dropna()
             .groupby(["repo", "gold_files"]).size().rename("tasks").reset_index()
             .sort_values("tasks", ascending=False).head(15))
    hot["share of repo"] = hot.apply(lambda r: r["tasks"] / (df["repo"] == r["repo"]).sum(), axis=1)
    display(Markdown("#### 🔥 Hot files: where reference fixes land most often"))
    display(hot.rename(columns={"gold_files": "file"}).style.hide(axis="index")
            .bar(subset=["tasks"], color="#FAD2CF").format({"share of repo": "{:.0%}"}))

### 🃏 Task card viewer
Call `show_task("<instance_id>")` on any task to see the issue, the stats and a colour-coded reference diff. Use it to build intuition and to debug your agent's trajectories.

In [ ]:
def render_diff(diff, max_lines=70):
    out = []
    for i, line in enumerate((diff or "").splitlines()):
        if i >= max_lines:
            out.append('<span style="color:#5F6368">… truncated …</span>'); break
        if line.startswith("+") and not line.startswith("+++"):
            style = "color:#137333;background:#E6F4EA"
        elif line.startswith("-") and not line.startswith("---"):
            style = "color:#C5221F;background:#FCE8E6"
        elif line.startswith("@@"):
            style = "color:#1A73E8;font-weight:600"
        elif line.startswith(("---", "+++", "diff ")):
            style = "color:#5F6368;font-weight:600"
        else:
            style = "color:#202124"
        out.append(f'<span style="{style}">{esc(line) or "&nbsp;"}</span>')
    return ('<pre style="font-size:12px;line-height:1.4;background:#F8F9FA;padding:10px;border-radius:8px;overflow-x:auto;margin:0">'
            + "\n".join(out) + "</pre>")

def badge(text, color="#1A73E8"):
    return f'<span style="background:{color};color:#fff;border-radius:10px;padding:2px 9px;margin-right:5px;font-size:12px">{esc(text)}</span>'

def show_task(instance_id, max_problem_chars=1800):
    r = df.loc[df["instance_id"] == instance_id]
    if r.empty:
        callout(f"Unknown instance_id <code>{esc(instance_id)}</code>", "err"); return
    r = r.iloc[0]
    problem = r["problem_statement"]
    problem = problem[:max_problem_chars] + ("\n…" if len(problem) > max_problem_chars else "")
    flags = "".join(badge(k.replace("_", " "), "#34A853") for k in
                    ["mentions_file_path", "mentions_symbol", "has_traceback", "has_hints"] if bool(r.get(k)))
    display(HTML(f"""
    <div style="border:1px solid #DADCE0;border-radius:14px;padding:16px 18px;color:#202124;background:#fff">
      <div style="font-size:20px;font-weight:700">{esc(r['instance_id'])}
        <span style="font-size:13px;color:#5F6368;font-weight:400">· {esc(r['repo'])} @ {esc(str(r['base_commit'])[:10])}</span></div>
      <div style="margin:8px 0">{badge(f"{r['n_src_files']} file(s)")}{badge(f"{r['n_hunks']} hunk(s)", "#A142F4")}
        {badge(f"+{r['added']} / -{r['deleted']}", "#EA4335")}{badge(str(r['size_bucket']), "#5F6368")}{flags}</div>
      <div style="font-size:12px;color:#5F6368">gold files: <code>{esc(', '.join(r['gold_files']))}</code>
        · symbols: <code>{esc(', '.join(r['gold_symbols']) or '—')}</code></div>
      <h4 style="margin:12px 0 4px 0">📝 Issue</h4>{pre(problem, 40)}
      <h4 style="margin:12px 0 4px 0">🩹 Reference patch</h4>{render_diff(r['patch'])}
      <details style="margin-top:8px"><summary style="cursor:pointer;font-weight:600">🧪 Test patch ({r['n_new_tests']} new test functions)</summary>{render_diff(r['test_patch'], 50)}</details>
    </div>"""))

median_task = df.iloc[(df["churn"] - df["churn"].median()).abs().argsort().iloc[0]]["instance_id"]
show_task(median_task)

<a id="graphs"></a>
# 🕸️ 6. Code intelligence: call graphs & embeddings
The dataset ships pre-computed code graphs and symbol embeddings. These are what the graph and search tools query. The file format is **not assumed**: the loaders below accept node-link JSON (`nodes` + `edges`/`links`), GraphML, pickled NetworkX graphs, `.npz`/`.npy` matrices and gzip variants, and they report what they actually found.

In [ ]:
_LISTING = {}
def list_files(subdir, max_depth=3):
    if subdir not in _LISTING:
        base = (DATA_DIR / subdir) if DATA_DIR else None
        _LISTING[subdir] = [p for p in limited_walk(base, max_depth) if p.is_file()] if base and base.exists() else []
    return _LISTING[subdir]

def find_artifact(subdir, row):
    """Best file in `subdir` for a task: match instance_id, then base_commit, then repo name."""
    files = list_files(subdir)
    if not files:
        return None
    repo = str(row.get("repo", ""))
    keys = [str(row["instance_id"]), str(row.get("base_commit", ""))[:12], str(row.get("base_commit", ""))[:7],
            repo.replace("/", "__"), repo.split("/")[-1]]
    for k in [k for k in keys if k and len(k) >= 4]:
        hits = [p for p in files if k in str(p.relative_to(DATA_DIR / subdir))]
        if hits:
            return sorted(hits, key=lambda p: len(str(p)))[0]
    return None

def load_any(path):
    path = Path(path)
    name = path.name.lower()
    opener = gzip.open if name.endswith(".gz") else open
    stem = name[:-3] if name.endswith(".gz") else name
    if stem.endswith(".json"):
        with opener(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    if stem.endswith(".jsonl"):
        with opener(path, "rt", encoding="utf-8") as f:
            return [json.loads(l) for l in f if l.strip()]
    if stem.endswith(".graphml"):
        return nx.read_graphml(path)
    if stem.endswith((".gpickle", ".pkl", ".pickle")):
        with opener(path, "rb") as f:        # competition-provided file; do not unpickle untrusted data
            return pickle.load(f)
    if stem.endswith(".npz"):
        return dict(np.load(path, allow_pickle=True))
    if stem.endswith(".npy"):
        return np.load(path, allow_pickle=True)
    if stem.endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError(f"unsupported format: {path.name}")

_SRC_KEYS, _DST_KEYS = ("source", "src", "from", "caller", "u"), ("target", "dst", "to", "callee", "v")
_TYPE_KEYS = ("type", "kind", "relation", "label", "edge_type", "rel")

def to_graph(obj):
    if isinstance(obj, nx.Graph):
        return nx.DiGraph(obj)
    if isinstance(obj, dict):
        for k in ("graph", "data", "code_graph"):
            if isinstance(obj.get(k), dict) and "nodes" in obj[k]:
                obj = obj[k]
        nodes, edges = obj.get("nodes"), obj.get("edges", obj.get("links"))
        if nodes is None or edges is None:
            return None
        G = nx.DiGraph()
        items = nodes.items() if isinstance(nodes, dict) else ((None, n) for n in nodes)
        for key, n in items:
            if isinstance(n, dict):
                nid = key if key is not None else next((n[k] for k in ("id", "name", "qualname", "qualified_name") if k in n), None)
                G.add_node(nid, **{k: v for k, v in n.items() if isinstance(v, (str, int, float, bool)) and k != "id"})
            else:
                G.add_node(key if key is not None else n)
        for e in edges:
            if isinstance(e, dict):
                s = next((e[k] for k in _SRC_KEYS if k in e), None); t = next((e[k] for k in _DST_KEYS if k in e), None)
                typ = next((e[k] for k in _TYPE_KEYS if k in e), "edge")
            elif isinstance(e, (list, tuple)) and len(e) >= 2:
                s, t = e[0], e[1]; typ = e[2] if len(e) > 2 and isinstance(e[2], str) else "edge"
            else:
                continue
            if s is not None and t is not None:
                G.add_edge(s, t, type=str(typ))
        return G
    return None

def node_tail(n):
    return re.split(r"[.:/#]", str(n))[-1]

def node_label(n):
    parts = [p for p in re.split(r"[.:/#]", str(n)) if p]
    return ".".join(parts[-2:]) if len(parts) > 1 else str(n)

def symbol_nodes(G, symbols):
    symbols = set(symbols)
    return [n for n in G.nodes if node_tail(n) in symbols or G.nodes[n].get("name") in symbols]

print("graph files:", len(list_files("graphs")), "· embedding files:", len(list_files("embeddings")),
      "· snapshot entries:", len(list((DATA_DIR / "snapshots").iterdir())) if DATA_DIR and (DATA_DIR / "snapshots").is_dir() else 0)

In [ ]:
_GRAPH_CACHE = {}
def graph_for(row):
    p = find_artifact("graphs", row)
    if p is None:
        return None, None
    if p not in _GRAPH_CACHE:
        _GRAPH_CACHE[p] = to_graph(load_any(p))
        if len(_GRAPH_CACHE) > 8:                 # keep memory bounded
            _GRAPH_CACHE.pop(next(iter(_GRAPH_CACHE)))
    return _GRAPH_CACHE[p], p

with safe("graph coverage scan"):
    t0, recs = time.time(), []
    for _, r in df.head(CFG["graph_scan_max_tasks"]).iterrows():
        if time.time() - t0 > CFG["time_budget_s"]:
            break
        G, p = graph_for(r)
        if G is None:
            continue
        recs.append({"instance_id": r["instance_id"], "nodes": G.number_of_nodes(), "edges": G.number_of_edges(),
                     "gold_symbol_in_graph": bool(r["gold_symbols"]) and bool(symbol_nodes(G, r["gold_symbols"])),
                     "has_gold_symbols": bool(r["gold_symbols"])})
    GRAPH_SCAN = pd.DataFrame(recs)
    if GRAPH_SCAN.empty:
        callout("No parseable code graphs found. Check the format in the inventory above and extend <code>to_graph</code>.", "warn")
    else:
        sub = GRAPH_SCAN[GRAPH_SCAN["has_gold_symbols"]]
        kpis([("Tasks scanned", len(GRAPH_SCAN)), ("Median nodes", int(GRAPH_SCAN["nodes"].median())),
              ("Median edges", int(GRAPH_SCAN["edges"].median())),
              ("Gold symbol is a graph node", f"{sub['gold_symbol_in_graph'].mean():.0%}" if len(sub) else "n/a")])

In [ ]:
def plot_fix_neighbourhood(instance_id=None, max_nodes=40):
    rows = df[df["gold_symbols"].str.len() > 0]
    row = rows[rows["instance_id"] == instance_id].iloc[0] if instance_id else None
    if row is None:
        for _, r in rows.iterrows():                       # first task whose gold symbol is in its graph
            G, _ = graph_for(r)
            if G is not None and symbol_nodes(G, r["gold_symbols"]):
                row = r; break
    if row is None:
        callout("No task with a gold symbol present in its graph.", "warn"); return
    G, path = graph_for(row)
    centers = symbol_nodes(G, row["gold_symbols"])
    center = max(centers, key=G.degree)
    und = G.to_undirected(as_view=True)
    ego = set(nx.ego_graph(und, center, radius=1).nodes)
    if len(ego) < 8:
        ego |= set(nx.ego_graph(und, center, radius=2).nodes)
    ego = sorted(ego, key=lambda n: (n != center, -G.degree(n)))[:max_nodes]
    H = G.subgraph(ego)

    etypes = sorted({d.get("type", "edge") for _, _, d in H.edges(data=True)})
    ecol = {t: PAL[(i + 4) % len(PAL)] for i, t in enumerate(etypes)}
    pos = nx.spring_layout(H, seed=CFG["seed"], k=1.2 / np.sqrt(max(len(H), 1)))
    fig, ax = plt.subplots(figsize=(13, 8))
    for t in etypes:
        el = [(u, v) for u, v, d in H.edges(data=True) if d.get("type", "edge") == t]
        nx.draw_networkx_edges(H, pos, edgelist=el, edge_color=ecol[t], arrows=True, arrowsize=12, width=1.4, alpha=.75, ax=ax,
                               connectionstyle="arc3,rad=0.08")
    others = [n for n in H if n != center]
    nx.draw_networkx_nodes(H, pos, nodelist=others, node_size=[250 + 60 * H.degree(n) for n in others],
                           node_color=PAL[0], alpha=.85, ax=ax)
    nx.draw_networkx_nodes(H, pos, nodelist=[center], node_size=1600, node_color=PAL[1], edgecolors="#202124", linewidths=2, ax=ax)
    nx.draw_networkx_labels(H, pos, labels={n: node_label(n) for n in H}, font_size=8, font_weight="bold", ax=ax)
    ax.legend(handles=[plt.Line2D([0], [0], color=ecol[t], lw=3, label=t) for t in etypes]
              + [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=PAL[1], markersize=13, label="symbol changed by the fix")],
              loc="upper left", frameon=True)
    ax.set_title(f"Where the fix lives: neighbourhood of `{node_tail(center)}` · {row['instance_id']}  "
                 f"({G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges in full graph)")
    ax.axis("off"); plt.tight_layout(); plt.show()
    top = sorted(G.nodes, key=lambda n: G.in_degree(n), reverse=True)[:8]
    display(pd.DataFrame({"most-referenced symbol": [node_tail(n) for n in top], "in-degree": [G.in_degree(n) for n in top],
                          "full id": [str(n) for n in top]}).style.hide(axis="index"))

with safe("fix neighbourhood plot"):
    plot_fix_neighbourhood()

In [ ]:
def unpack_embeddings(obj):
    """Return (ids | None, matrix, summary DataFrame) from npz dict / array / DataFrame."""
    arrays = obj if isinstance(obj, dict) else {"array": obj}
    if isinstance(obj, pd.DataFrame):
        num = obj.select_dtypes("number")
        arrays = {"matrix": num.values, **{c: obj[c].values for c in obj.columns if c not in num.columns}}
    summary = pd.DataFrame([{"key": k, "shape": tuple(np.shape(v)), "dtype": str(np.asarray(v).dtype)} for k, v in arrays.items()])
    mats = {k: np.asarray(v) for k, v in arrays.items() if np.asarray(v).ndim == 2 and np.asarray(v).dtype.kind == "f"}
    if not mats:
        return None, None, summary
    vk = max(mats, key=lambda k: mats[k].shape[0])
    V = mats[vk].astype(np.float32)
    ids = next((np.asarray(v).astype(str) for k, v in arrays.items()
                if k != vk and np.asarray(v).ndim == 1 and len(v) == len(V) and np.asarray(v).dtype.kind in "USO"), None)
    return ids, V, summary

def module_of(sym):
    s = str(sym)
    if ":" in s:
        return s.split(":")[0]
    return s.rsplit(".", 1)[0] if "." in s else "(root)"

with safe("embedding explorer"):
    row = next((r for _, r in df.iterrows() if find_artifact("embeddings", r) is not None and r["gold_symbols"]), None)
    if row is None:
        callout("No embedding files found for any task.", "warn")
    else:
        epath = find_artifact("embeddings", row)
        ids, V, summ = unpack_embeddings(load_any(epath))
        display(Markdown(f"**{epath.relative_to(DATA_DIR)}** contains:")); display(summ.style.hide(axis="index"))
        if V is None:
            callout("No float matrix found in this file.", "warn")
        else:
            n = len(V)
            idx = np.random.default_rng(CFG["seed"]).choice(n, size=min(n, 6000), replace=False)
            Xc = V[idx] - V[idx].mean(0)
            _, _, vt = np.linalg.svd(Xc, full_matrices=False)
            P = Xc @ vt[:2].T
            kpis([("Vectors", f"{n:,}"), ("Dimension", V.shape[1]), ("Mean L2 norm", f"{np.linalg.norm(V, axis=1).mean():.2f}")])
            fig, ax = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.5, 1]})
            if ids is not None:
                mods = pd.Series([module_of(ids[i]) for i in idx])
                top_mods = mods.value_counts().index[:7].tolist()
                for j, m in enumerate(top_mods):
                    mask = (mods == m).values
                    ax[0].scatter(P[mask, 0], P[mask, 1], s=12, alpha=.7, color=PAL[j], label=textwrap.shorten(m, 40))
                rest = ~mods.isin(top_mods).values
                ax[0].scatter(P[rest, 0], P[rest, 1], s=8, alpha=.3, color="#BDC1C6", label="other")
                gold_pos = [k for k, i in enumerate(idx) if node_tail(ids[i]) in set(row["gold_symbols"])]
                for k in gold_pos:
                    ax[0].scatter(P[k, 0], P[k, 1], s=300, marker="*", color=PAL[1], edgecolors="#202124", zorder=5)
                    ax[0].annotate(node_tail(ids[idx[k]]), P[k], xytext=(6, 6), textcoords="offset points", fontweight="bold")
                ax[0].legend(fontsize=8, markerscale=2, loc="best")
            else:
                ax[0].scatter(P[:, 0], P[:, 1], s=10, alpha=.6)
            ax[0].set_title(f"Symbol embeddings (PCA) · {row['instance_id']} · ★ = changed by the fix")
            norms = np.linalg.norm(V, axis=1)
            ax[1].hist(norms, bins=40, color=PAL[3], edgecolor="white"); ax[1].set_title("L2 norm distribution")
            plt.tight_layout(); plt.show()

            if ids is not None:
                gold = [i for i, s in enumerate(ids) if node_tail(s) in set(row["gold_symbols"])]
                if gold:
                    Vn = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
                    sims = Vn @ Vn[gold[0]]
                    order = [i for i in np.argsort(-sims) if i != gold[0]][:8]
                    display(Markdown(f"**Nearest neighbours of `{ids[gold[0]]}`**: roughly what similarity search would surface around the fix:"))
                    display(pd.DataFrame({"symbol": ids[order], "cosine": sims[order].round(3)}).style.hide(axis="index")
                            .bar(subset=["cosine"], color="#AECBFA").format({"cosine": "{:.3f}"}))

<a id="baseline"></a>
# 🎯 7. Localization baseline: can plain retrieval find the file?
Before spending GPU time on an agent, measure the difficulty of step 1. For each task we index every non-test `.py` file of the repository snapshot with **TF-IDF over code-aware tokens** (`get_request_handler` → `get request handler`, `CamelCase` → `camel case`) and rank files against the issue text.

- **TF-IDF**: pure lexical similarity.
- **TF-IDF + mention boost**: extra score when the issue explicitly names the file/module or a symbol *defined* in the file. This is what a disciplined `grep` step gives an agent.

**Hit@k** = share of tasks where a gold source file is in the top-k. A low Hit@5 means the agent needs real exploration (graph tools, sub-agent), not just reading the top result.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

_ID_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]+")
_PART_RE = re.compile(r"[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+")
def code_tokens(text):
    out = []
    for tok in _ID_RE.findall(text):
        out.append(tok.lower())
        parts = [p.lower() for p in _PART_RE.findall(tok) if len(p) > 1]
        if len(parts) > 1:
            out.extend(parts)
    return out

def strip_common_prefix(files):
    keys = list(files)
    if len(keys) > 1:
        first = {k.split("/", 1)[0] for k in keys}
        if len(first) == 1 and all("/" in k for k in keys):
            return strip_common_prefix({k.split("/", 1)[1]: v for k, v in files.items()})
    return files

def read_py_files(entry, max_files=8000, max_bytes=400_000):
    files = {}
    if entry.is_dir():
        for p in entry.rglob("*.py"):
            try:
                if p.stat().st_size <= max_bytes:
                    files[str(p.relative_to(entry))] = p.read_text(encoding="utf-8", errors="ignore")
            except OSError:
                pass
            if len(files) >= max_files:
                break
    elif zipfile.is_zipfile(entry):
        with zipfile.ZipFile(entry) as z:
            for i in z.infolist():
                if i.filename.endswith(".py") and i.file_size <= max_bytes and len(files) < max_files:
                    files[i.filename] = z.read(i).decode("utf-8", "ignore")
    elif tarfile.is_tarfile(entry):
        with tarfile.open(entry) as t:
            for m in t:
                if m.isfile() and m.name.endswith(".py") and m.size <= max_bytes and len(files) < max_files:
                    files[m.name] = t.extractfile(m).read().decode("utf-8", "ignore")
    return strip_common_prefix(files)

def resolve_snapshot(row):
    base = DATA_DIR / "snapshots" if DATA_DIR else None
    if not base or not base.exists():
        return None
    entries = [p for p in limited_walk(base, 1)]
    repo = str(row["repo"])
    for key in [row["instance_id"], str(row["base_commit"])[:12], str(row["base_commit"])[:7], repo.replace("/", "__"), repo.split("/")[-1]]:
        if key and len(key) >= 4:
            hits = [p for p in entries if key in p.name]
            if hits:
                return sorted(hits, key=lambda p: (len(p.relative_to(base).parts), len(p.name)))[0]
    return None

_DEF_RE = re.compile(r"^\s*(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)", re.M)
_INDEX_CACHE = {}
def build_index(entry):
    if entry not in _INDEX_CACHE:
        files = {k: v for k, v in read_py_files(entry).items() if not is_test_path(k)}
        paths = sorted(files)
        if not paths:
            _INDEX_CACHE[entry] = None
        else:
            docs = [(" ".join([p.replace("/", " ")] * 3)) + "\n" + files[p][:200_000] for p in paths]
            vec = TfidfVectorizer(tokenizer=code_tokens, lowercase=False, token_pattern=None, sublinear_tf=True, max_features=300_000)
            X = vec.fit_transform(docs)
            defs = [set(_DEF_RE.findall(files[p])) for p in paths]
            _INDEX_CACHE[entry] = (paths, vec, X, defs)
        if len(_INDEX_CACHE) > 6:
            _INDEX_CACHE.pop(next(iter(_INDEX_CACHE)))
    return _INDEX_CACHE[entry]

def rank_files(index, query, boost=False):
    paths, vec, X, defs = index
    scores = (X @ vec.transform([query]).T).toarray().ravel()
    if boost:
        q = query.lower()
        q_ids = {t for t in _ID_RE.findall(query) if len(t) >= 4 and ("_" in t or any(c.isupper() for c in t[1:]))}
        for i, p in enumerate(paths):
            stem = Path(p).stem
            if p.lower() in q or (stem not in GENERIC_STEMS and len(stem) >= 4 and re.search(r"\b" + re.escape(stem.lower()) + r"\b", q)):
                scores[i] += 1.0
            scores[i] += min(1.0, 0.35 * len(q_ids & defs[i]))
    return [paths[i] for i in np.argsort(-scores)]

def gold_rank(ranked, gold):
    for k, p in enumerate(ranked, 1):
        if any(p == g or p.endswith("/" + g) or g.endswith("/" + p) for g in gold):
            return k
    return None

with safe("localization baseline"):
    t0, recs = time.time(), []
    for _, r in df.head(CFG["localization_max_tasks"]).iterrows():
        if time.time() - t0 > CFG["time_budget_s"] or not r["gold_files"]:
            continue
        entry = resolve_snapshot(r)
        index = build_index(entry) if entry is not None else None
        if index is None:
            continue
        query = r["problem_statement"] + "\n" + r["hints_text"]
        recs.append({"instance_id": r["instance_id"], "repo": r["repo"], "n_files": len(index[0]),
                     "rank_tfidf": gold_rank(rank_files(index, query), r["gold_files"]),
                     "rank_boost": gold_rank(rank_files(index, query, boost=True), r["gold_files"])})
    LOC = pd.DataFrame(recs)
    if LOC.empty:
        callout("Could not index any repository snapshot (unknown layout?). Inspect <code>snapshots/</code> in the inventory "
                "and adapt <code>resolve_snapshot</code> / <code>read_py_files</code>.", "warn")
    else:
        ks = [1, 3, 5, 10, 20]
        hit = lambda col, k: (LOC[col].fillna(1e9) <= k).mean()
        fig, ax = plt.subplots(1, 2, figsize=(16, 5.2))
        for col, lab, c in [("rank_tfidf", "TF-IDF", PAL[0]), ("rank_boost", "TF-IDF + mention boost", PAL[1])]:
            ax[0].plot(ks, [hit(col, k) * 100 for k in ks], marker="o", lw=2.5, color=c, label=lab)
            for k in ks:
                ax[0].annotate(f"{hit(col, k):.0%}", (k, hit(col, k) * 100), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8, color=c)
        ax[0].set_xscale("log"); ax[0].set_xticks(ks, [str(k) for k in ks]); ax[0].set_ylim(0, 112)
        ax[0].set_xlabel("k (files shown to the agent)"); ax[0].set_ylabel("Hit@k (%)"); ax[0].legend()
        ax[0].set_title(f"How often is the gold file in the top-k?  ({len(LOC)} tasks)")
        per = LOC.groupby("repo").apply(lambda g: pd.Series({"Hit@1": (g["rank_boost"].fillna(1e9) <= 1).mean(),
                                                            "Hit@5": (g["rank_boost"].fillna(1e9) <= 5).mean(),
                                                            "Hit@10": (g["rank_boost"].fillna(1e9) <= 10).mean()})) * 100
        per.plot(kind="barh", ax=ax[1], color=[PAL[3], PAL[2], PAL[4]], width=.8)
        ax[1].set_xlim(0, 100); ax[1].set_xlabel("%"); ax[1].set_ylabel(""); ax[1].set_title("Boosted retriever per repository")
        plt.tight_layout(); plt.show()
        kpis([("Tasks evaluated", len(LOC)), ("Median repo size (py files)", int(LOC["n_files"].median())),
              ("Hit@1 (boost)", f"{hit('rank_boost', 1):.0%}"), ("Hit@5 (boost)", f"{hit('rank_boost', 5):.0%}"),
              ("MRR (boost)", f"{(1 / LOC['rank_boost'].dropna()).sum() / len(LOC):.2f}")])
        gain = hit("rank_boost", 5) - hit("rank_tfidf", 5)
        callout(f"Explicit mentions add <b>{gain:+.0%}</b> Hit@5 on top of pure TF-IDF. "
                "The system prompt below turns this into a rule: <i>grep for every identifier in the issue before reading files.</i>", "info")

<a id="edit"></a>
# ✂️ 8. Edit mechanics & patch safety
### A string-replacement `edit_file`, simulated
Agents lose a lot of turns to failed or *silently broken* edits. The simulator below re-implements the common three-tier strategy (**exact → whitespace-flexible → token regex**) and, crucially, **checks that the result still parses**. It is a local model for building intuition, and the real harness implementation is authoritative.

In [ ]:
import ast

def simulate_edit(content, old, new):
    """Return (new_content | None, tier, error | None)."""
    c = content.count(old) if old else 0
    if c == 1:
        return content.replace(old, new), "exact", None
    if c > 1:
        return None, "exact", f"old_string matches {c} times, so it must be unique"
    lines = content.split("\n")
    target = [l.strip() for l in old.strip("\n").split("\n")]
    if any(target):
        hits = [i for i in range(len(lines) - len(target) + 1)
                if all(lines[i + j].strip() == target[j] for j in range(len(target)))]
        if len(hits) == 1:
            i = hits[0]
            indent = lines[i][: len(lines[i]) - len(lines[i].lstrip())]
            block = [(indent + l) if l.strip() else "" for l in textwrap.dedent(new.strip("\n")).split("\n")]
            return "\n".join(lines[:i] + block + lines[i + len(target):]), "flexible", None
        if len(hits) > 1:
            return None, "flexible", f"{len(hits)} whitespace-insensitive matches"
    toks = old.split()
    if toks:
        ms = list(re.finditer(r"\s+".join(map(re.escape, toks)), content))
        if len(ms) == 1:
            return content[: ms[0].start()] + new + content[ms[0].end():], "regex", None
        if len(ms) > 1:
            return None, "regex", f"{len(ms)} regex matches"
    return None, "—", "old_string not found"

def compiles(src):
    try:
        ast.parse(src); return True, ""
    except SyntaxError as e:
        return False, f"{type(e).__name__}: {e.msg} (line {e.lineno})"

ORIGINAL = """def calculate_total(items):
    total = 0
    for item in items:
        total += item.price
    return total
"""
CASES = [
    ("Exact match, indentation included", "        total += item.price",
     "        if item.is_valid:\n            total += item.price"),
    ("Exact match, indentation missing (classic pitfall)", "total += item.price",
     "if item.is_valid:\n    total += item.price"),
    ("Wrong indentation in old_string → flexible tier", "for item in items:\n    total += item.price",
     "for item in items:\n    if item.is_valid:\n        total += item.price"),
    ("Collapsed whitespace → regex tier", "total  +=   item.price", "total += item.price * item.qty"),
    ("Ambiguous old_string", "total", "grand_total"),
]
rows, outputs = [], {}
for name, old, new in CASES:
    out, tier, err = simulate_edit(ORIGINAL, old, new)
    ok, why = compiles(out) if out is not None else (None, "")
    outputs[name] = out
    rows.append({"case": name, "tier": tier, "applied": "✅" if out is not None else "❌",
                 "still parses": "✅" if ok else ("💥 " + why if ok is False else "—"), "error": err or ""})
display(pd.DataFrame(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))
details("👀 See the silently broken result of the classic pitfall", pre(outputs[CASES[1][0]]))
callout("<b>Lesson baked into the prompt below:</b> copy <code>old_string</code> verbatim <i>including leading whitespace</i>, "
        "keep it short but unique, and run <code>python -m py_compile &lt;file&gt;</code> after every edit. "
        "An edit that is applied but no longer parses is invisible until the hidden tests fail.", "info")

### 🧪 Patch sanity checker
Run this on every patch your agent produces during local experiments. On the reference patches it doubles as a check of the parser itself.

In [ ]:
PROTECTED = ("conftest.py", "pytest.ini", "tox.ini", "noxfile.py", "setup.cfg", "pyproject.toml", ".github/")
SCRATCH_RE = re.compile(r"(^|/)(repro|reproduce|scratch|debug|tmp|temp)[^/]*\.py$", re.I)

def check_patch(diff, max_churn=400):
    """Return a list of (severity, message). severity ∈ {'error', 'warn'}."""
    if not diff or not diff.strip():
        return [("error", "empty patch")]
    info, issues = parse_patch(diff), []
    if not info["files"]:
        issues.append(("error", "no file headers found"))
    if info["malformed"] or info["unterminated"]:
        issues.append(("error", "hunk line counts do not match the @@ headers (corrupted diff)"))
    churn = 0
    for f in info["files"]:
        churn += f["added"] + f["deleted"]
        if is_test_path(f["path"]):
            issues.append(("warn", f"modifies a test file: {f['path']}"))
        if any(f["path"].endswith(p) or p in f["path"] for p in PROTECTED):
            issues.append(("warn", f"touches config/harness file: {f['path']}"))
        if f["status"] == "added" and SCRATCH_RE.search(f["path"]):
            issues.append(("error", f"looks like a leftover scratch file: {f['path']}"))
    if churn > max_churn:
        issues.append(("warn", f"very large patch ({churn} changed lines)"))
    return issues

gold_issues = Counter(msg.split(":")[0] for d in df["patch"] for _, msg in check_patch(d))
clean = sum(1 for d in df["patch"] if not check_patch(d))
kpis([("Reference patches checked", len(df)), ("Clean", f"{clean / max(len(df), 1):.0%}")])
if gold_issues:
    display(pd.DataFrame(gold_issues.most_common(), columns=["finding on reference patches", "count"]).style.hide(axis="index"))

demo_bad = "--- a/pkg/core.py\n+++ b/pkg/core.py\n@@ -1 +1 @@\n-x = 1\n+x = 2\n--- /dev/null\n+++ b/repro_issue.py\n@@ -0,0 +1,1 @@\n+print('debug')\n"
print("Example agent patch →", check_patch(demo_bad))

<a id="prompts"></a>
# 🧠 9. Prompts & context budget
The prompts are **generated from `AVAILABLE_TOOLS`**, so they never reference a tool that does not exist. They encode the lessons from the sections above: grep-first localization, reproduce before fixing, whitespace-exact edits, compile checks, budget discipline, and a clean diff.

In [ ]:
def build_system_prompt(architecture, tools=None):
    tools = set(tools if tools is not None else AVAILABLE_TOOLS)
    T = lambda name: name in tools
    loc = []
    if architecture == "analyzer+coder":
        loc.append("Call the `code_analyzer` tool with the full issue text first. It returns LOCATION / ROOT CAUSE / FIX PLAN. Verify its claim by reading those exact lines before editing.")
    if T("run_command"):
        loc.append("Extract every identifier, error message and file name from the issue and search for them: `grep -rn \"<identifier>\" --include=*.py . | head -30`.")
    if T("search_similar_code") and architecture == "single":
        loc.append("When the issue describes behaviour without naming code, use `search_similar_code` with a short natural-language query.")
    if T("get_code_neighbors") and architecture == "single":
        loc.append("Use `get_code_neighbors` to follow callers/callees from a candidate function to the place where behaviour diverges.")
    how = (["`read_file` with a line range"] if T("read_file") else []) + (["`sed -n 'START,ENDp' FILE`"] if T("run_command") else [])
    loc.append("Read only the lines you need" + (f" ({' or '.join(how)})." if how else "."))
    loc_txt = "\n".join(f"   - {l}" for l in loc)
    edit_how = ("edit source files with `edit_file`. Copy `old_string` verbatim from the file, *including leading indentation*, "
                "and strip any line-number prefixes. Keep `old_string` short but unique."
                if T("edit_file") else
                "edit source files with a small Python snippet run via the shell (read the file, `str.replace` one unique, "
                "whitespace-exact snippet, write it back) and assert that the snippet occurs exactly once.")
    status = ("- Call `get_status` every ~8 tool calls. When less than 25% of turns or time remain, stop exploring and go straight to Fix → Verify → Submit.\n"
              if T("get_status") else "- You have a limited number of turns: stop exploring after ~60% of your effort and move to Fix → Verify → Submit.\n")
    return f"""You are an autonomous senior Python engineer working inside a sandboxed checkout of a real open-source repository at /workspace.
Goal: resolve the issue in the user message with the smallest correct patch, then call `submit_patch`.

## Hard rules
- Never edit, add or delete tests, `conftest.py`, `pytest.ini`, CI or packaging files. Hidden tests are applied after you finish.
- Keep public APIs backward compatible unless the issue explicitly asks for a change.
- Scratch files go to /tmp only. Anything left in /workspace becomes part of your patch.
- The environment is pre-built: do not try to install packages.
- Always finish by calling `submit_patch`. A careful best-effort fix beats no patch.

## Workflow
1. **Understand**: state the expected vs. actual behaviour to yourself in one or two sentences.
2. **Localize**:
{loc_txt}
3. **Reproduce**: write a minimal script to /tmp/repro.py that shows the bug and run it with `python /tmp/repro.py`.
4. **Fix**: {edit_how} One logical change per edit. Fix the root cause, not the symptom, and also handle the edge cases the issue mentions.
5. **Verify**: run `python -m py_compile <file>` after every edit, rerun /tmp/repro.py, then run the closest existing tests: `python -m pytest <tests/path> -x -q` (narrow with `-k`).
6. **Submit**: run `git status` and `git diff`, make sure only intended source changes remain, then call `submit_patch`.

## Budget discipline
{status}- Keep outputs short: pipe through `head`, use `grep -n`, `pytest -q`. Never print whole large files.
- If an edit fails twice, re-read the exact lines and retry with a smaller unique snippet.

## Quality bar
- Match the surrounding code style, type hints and naming.
- Prefer a small, targeted change over a refactor. Touch other files only when the fix requires it.
"""

def build_analyzer_prompt(available=None):
    available = set(available if available is not None else AVAILABLE_TOOLS)
    T = lambda name: name in available
    tools = []
    if T("run_command"):
        tools.append("`run_command` for READ-ONLY commands only: `grep -rn`, `ls`, `sed -n`, `git log -p -S`")
    if T("search_similar_code"):
        tools.append("`search_similar_code` for concepts the issue describes without naming code")
    if T("get_code_neighbors"):
        tools.append("`get_code_neighbors` to walk callers and callees")
    if T("get_code_subgraph"):
        tools.append("`get_code_subgraph` to see how a few candidate symbols connect")
    if T("read_file"):
        tools.append("`read_file` with tight line ranges to confirm")
    tools_txt = "\n".join(f"- {t}" for t in tools)
    return f"""You are `code_analyzer`, a read-only code navigation specialist. You never modify files.
Given an issue, find exactly where it must be fixed.

## Tools
{tools_txt}

## Method
1. Extract identifiers from the issue: function/class names, error messages, file paths, options.
2. Search for each one, then follow the call chain until you reach the line where behaviour diverges from what the issue expects.
3. Confirm by reading the actual code. Never guess line numbers.

## Answer format (at most 250 words, nothing else)
LOCATION: <path>:<start>-<end> (<function or class>)
ROOT CAUSE: <one or two sentences>
FIX PLAN: <concrete change>
RELATED: <other call sites or files needing the same change, or "none">
TESTS: <existing test files that exercise this code>
CONFIDENCE: high | medium | low
"""

ARCH = CFG["architecture"] if CFG["architecture"] in ("single", "analyzer+coder") else "analyzer+coder"
SYSTEM_PROMPT = build_system_prompt(ARCH)
ANALYZER_PROMPT = build_analyzer_prompt() if ARCH == "analyzer+coder" else ""
details("📝 prompts/system.md", pre(SYSTEM_PROMPT, 120))
if ANALYZER_PROMPT:
    details("🔍 prompts/analyzer.md", pre(ANALYZER_PROMPT, 120))

In [ ]:
est_tokens = lambda s: int(len(s) / 3.6)            # rough heuristic for English + code, not a real tokenizer

with safe("context budget"):
    tool_schema_tokens = 130 * (len(AVAILABLE_TOOLS) + (1 if ARCH == "analyzer+coder" else 0))
    issue_tok = df["problem_statement"].map(est_tokens)
    scen = {"median issue": int(issue_tok.median()), "p90 issue": int(issue_tok.quantile(.9)), "max issue": int(issue_tok.max())}
    comp_names = ["system prompt", "tool schemas", "issue text", "reserved for output"]
    colors = [PAL[0], PAL[4], PAL[2], PAL[1]]
    fig, ax = plt.subplots(figsize=(14, 3.6))
    for yi, (label, itok) in enumerate(scen.items()):
        parts = [est_tokens(SYSTEM_PROMPT), tool_schema_tokens, itok, CFG["max_output_tokens"]]
        left = 0
        for name, v, c in zip(comp_names, parts, colors):
            ax.barh(yi, v, left=left, color=c, label=name if yi == 0 else None, edgecolor="white")
            left += v
        free = CONTEXT_LIMIT - left
        ax.barh(yi, max(free, 0), left=left, color="#E8EAED", label="left for the trajectory" if yi == 0 else None, edgecolor="white")
        ax.text(min(left, CONTEXT_LIMIT) + 200, yi, f"{max(free, 0):,} tok free ({max(free, 0) / CONTEXT_LIMIT:.0%})", va="center", fontsize=9)
    ax.axvline(CONTEXT_LIMIT, color="#202124", ls="--"); ax.set_yticks(range(len(scen)), list(scen))
    ax.set_xlim(0, CONTEXT_LIMIT * 1.08); ax.set_xlabel("tokens (approx.)")
    ax.set_title(f"Where does a {CONTEXT_LIMIT:,}-token context go before the first tool call?")
    ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(.5, -.28), frameon=False)
    plt.tight_layout(); plt.show()
    if CFG["max_output_tokens"] > 0.35 * CONTEXT_LIMIT:
        callout(f"<code>max_output_tokens={CFG['max_output_tokens']:,}</code> reserves more than 35% of the context. "
                "Every tool result competes for the remainder, so consider lowering it.", "warn")
    callout("Each <code>read_file</code> or command output lands in this grey area. That is why the analyzer sub-agent helps: "
            "its exploration happens in <i>its own</i> context and only a short report comes back to the coder.", "info")

<a id="bundle"></a>
# 🏗️ 10. Agent bundle builder
**Step 1: learn from the official sample.** We locate `sample_submission` (as a folder or a zip), show its files, and detect the model alias, root file name and whether it uses `!include`.

In [ ]:
ROOT_NAMES = ["agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml"]

class TolerantLoader(yaml.SafeLoader):
    """SafeLoader that keeps custom tags such as !include as {'__tag__': '!include', 'value': ...}."""
def _tagged(loader, suffix, node):
    if isinstance(node, yaml.ScalarNode):
        val = loader.construct_scalar(node)
    elif isinstance(node, yaml.SequenceNode):
        val = loader.construct_sequence(node, deep=True)
    else:
        val = loader.construct_mapping(node, deep=True)
    return {"__tag__": "!" + suffix, "value": val}
TolerantLoader.add_multi_constructor("!", _tagged)

def load_yaml(path):
    return yaml.load(Path(path).read_text(encoding="utf-8"), Loader=TolerantLoader)

def find_sample_submission():
    if DEMO_MODE:
        return None
    for c in sorted(p for p in DATA_DIR.iterdir() if "sample" in p.name.lower()):
        if c.is_dir():
            ymls = sorted((p for p in c.rglob("*") if p.suffix in (".yaml", ".yml")), key=lambda p: len(p.parts))
            if ymls:
                return ymls[0].parent
            zips = sorted(c.rglob("*.zip"))
            if zips:
                c = zips[0]
        if c.is_file() and zipfile.is_zipfile(c):
            out = WORK / "_sample_submission"
            shutil.rmtree(out, ignore_errors=True)
            with zipfile.ZipFile(c) as z:
                z.extractall(out)
            ymls = sorted((p for p in out.rglob("*") if p.suffix in (".yaml", ".yml")), key=lambda p: len(p.parts))
            return ymls[0].parent if ymls else out
    return None

SAMPLE_ROOT = find_sample_submission()
SAMPLE = {"root_file": None, "models": set(), "uses_include": True, "top_keys": [], "text": ""}
if SAMPLE_ROOT:
    ytexts = {p: p.read_text(encoding="utf-8", errors="ignore") for p in SAMPLE_ROOT.rglob("*") if p.suffix in (".yaml", ".yml")}
    SAMPLE["text"] = "\n".join(ytexts.values())
    SAMPLE["root_file"] = next((SAMPLE_ROOT / n for n in ROOT_NAMES if (SAMPLE_ROOT / n).exists()), None)
    SAMPLE["models"] = {m for t in ytexts.values() for m in re.findall(r"^\s*model:\s*['\"]?([^'\"\s#]+)", t, flags=re.M)}
    SAMPLE["uses_include"] = "!include" in SAMPLE["text"]
    with safe("sample root parsing"):
        if SAMPLE["root_file"]:
            SAMPLE["top_keys"] = list((load_yaml(SAMPLE["root_file"]) or {}).keys())
    files = sorted(p for p in SAMPLE_ROOT.rglob("*") if p.is_file())
    display(Markdown(f"**sample submission** at `{SAMPLE_ROOT}` · {len(files)} files"))
    for p in files[:25]:
        if p.suffix.lower() in TEXT_EXTS:
            details(f"📄 {p.relative_to(SAMPLE_ROOT)} ({human_bytes(p.stat().st_size)})",
                    pre(p.read_text(encoding="utf-8", errors="ignore"), 60))
else:
    callout("No sample submission found. The bundle will be built from this notebook's templates only.", "warn")

if CFG["model_alias"]:
    MODEL_ALIAS, model_src = CFG["model_alias"], "CFG override"
elif len(SAMPLE["models"]) == 1:
    MODEL_ALIAS, model_src = next(iter(SAMPLE["models"])), "sample submission"
else:
    MODEL_ALIAS, model_src = CFG["fallback_model_alias"], "fallback (verify in HARNESS_README!)"
INCLUDE_STYLE = "include" if SAMPLE["uses_include"] else "inline"
ROOT_FILE_NAME = SAMPLE["root_file"].name if SAMPLE["root_file"] else "agent.yaml"
kpis([("Model alias", MODEL_ALIAS), ("alias source", model_src), ("Root file", ROOT_FILE_NAME),
      ("Prompt style", INCLUDE_STYLE), ("Architecture", ARCH)])

**Step 2: write the bundle.** Two architectures are available:

| | `single` | `analyzer+coder` *(default)* |
|:--|:--|:--|
| Agents | one coder with every tool | coder + read-only `code_analyzer` wrapped as a tool |
| Context | exploration fills the coder's context | exploration stays in the analyzer's context, and only a short report returns |
| Best when | short issues, strong hints | vague issues, large repos |

With `bundle_base="sample"`, the official sample is copied untouched and **only its instruction is replaced** with our system prompt. This is the most conservative option if the validator reports schema differences.

In [ ]:
BUNDLE = WORK / CFG["bundle_dir"]
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True)

SAMPLING_YAML = f"""temperature: {CFG['temperature']}
top_p: {CFG['top_p']}
top_k: {CFG['top_k']}
max_output_tokens: {CFG['max_output_tokens']}
thinking_config:
  thinking_budget: {CFG['thinking_budget']}
  include_thoughts: false
"""

def field_instruction(rel_path, text):
    if INCLUDE_STYLE == "include":
        return f"instruction: !include {rel_path}"
    return "instruction: |\n" + textwrap.indent(text.rstrip() + "\n", "  ").rstrip()

def field_sampling(rel_path):
    if INCLUDE_STYLE == "include":
        return f"generate_content_config: !include {rel_path}"
    return "generate_content_config:\n" + textwrap.indent(SAMPLING_YAML, "  ").rstrip()

def write(rel, text):
    p = BUNDLE / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text.rstrip() + "\n", encoding="utf-8")

T = lambda name: name in AVAILABLE_TOOLS
CODER_TOOLS = [t for t in ["run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch"] if T(t)]
if ARCH == "single":
    CODER_TOOLS += [t for t in ["search_similar_code", "get_code_neighbors", "get_code_subgraph"] if T(t)]
ANALYZER_TOOLS = [t for t in ["run_command", "read_file", "search_similar_code", "get_code_neighbors", "get_code_subgraph"] if T(t)]
if "submit_patch" not in CODER_TOOLS:
    callout("<code>submit_patch</code> was not verified in the harness files, so the agent may be unable to finish. Check the README.", "err")

if CFG["bundle_base"] == "sample" and SAMPLE_ROOT and SAMPLE["root_file"]:
    shutil.copytree(SAMPLE_ROOT, BUNDLE, dirs_exist_ok=True)
    sample_tools = [t for t in ((load_yaml(SAMPLE["root_file"]) or {}).get("tools") or []) if isinstance(t, str)]
    SYSTEM_PROMPT = build_system_prompt("single", tools=sample_tools)     # only mention tools this agent really has
    not_used = [t for t in CODER_TOOLS if t not in sample_tools]
    if not_used:
        callout("The sample agent does not declare these verified tools: " + ", ".join(f"<code>{t}</code>" for t in not_used)
                + ". The prompt was adapted to the sample's tools. Adding them is a likely easy win.", "info")
    root_path = BUNDLE / ROOT_FILE_NAME
    lines = root_path.read_text(encoding="utf-8").split("\n")
    idx = next((i for i, l in enumerate(lines) if l.startswith("instruction:")), None)
    if idx is None:
        callout("Sample root config has no top-level <code>instruction:</code>. Prompt not replaced.", "err")
    else:
        m = re.match(r"instruction:\s*!include\s+(\S+)", lines[idx])
        if m:
            (root_path.parent / m.group(1)).write_text(SYSTEM_PROMPT, encoding="utf-8")
        else:
            end = idx + 1
            while end < len(lines) and (lines[end].startswith((" ", "\t")) or not lines[end].strip()):
                end += 1
            lines[idx:end] = ["instruction: |", *textwrap.indent(SYSTEM_PROMPT.rstrip(), "  ").split("\n")]
            root_path.write_text("\n".join(lines), encoding="utf-8")
    callout("Bundle = official sample + our system prompt.", "ok")
else:
    write("prompts/system.md", SYSTEM_PROMPT)
    write("configs/sampling.yaml", SAMPLING_YAML)
    tools_yaml = "\n".join(f"  - {t}" for t in CODER_TOOLS)
    if ARCH == "analyzer+coder":
        write("prompts/analyzer.md", ANALYZER_PROMPT)
        write("sub_agents/code_analyzer.yaml", "\n".join([
            "name: code_analyzer",
            f"model: {MODEL_ALIAS}",
            "description: Read-only code navigator. Given an issue, returns the exact location, root cause and a fix plan.",
            field_instruction("../prompts/analyzer.md", ANALYZER_PROMPT),
            field_sampling("../configs/sampling.yaml"),
            "tools:", *[f"  - {t}" for t in ANALYZER_TOOLS]]))
        tools_yaml += "\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true"
    write(ROOT_FILE_NAME, "\n".join([
        "name: swe_coder",
        f"model: {MODEL_ALIAS}",
        "description: Autonomous software engineer that fixes repository issues with minimal, verified patches.",
        field_instruction("prompts/system.md", SYSTEM_PROMPT),
        field_sampling("configs/sampling.yaml"),
        "tools:", tools_yaml]))

def tree_lines(d, prefix=""):
    kids = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
    out = []
    for i, k in enumerate(kids):
        last = i == len(kids) - 1
        out.append(f"{prefix}{'└── ' if last else '├── '}{k.name}{'/' if k.is_dir() else f'  ({human_bytes(k.stat().st_size)})'}")
        if k.is_dir():
            out += tree_lines(k, prefix + ("    " if last else "│   "))
    return out

def show_bundle(root):
    files = sorted(p for p in root.rglob("*") if p.is_file())
    display(HTML(pre("\n".join([f"{root.name}/", *tree_lines(root)]))))
    for p in files:
        if p.suffix in (".yaml", ".yml"):
            details(f"📄 {p.relative_to(root)}", pre(p.read_text(encoding="utf-8")))

show_bundle(BUNDLE)

if SAMPLE["top_keys"] and CFG["bundle_base"] != "sample":
    ours = list((load_yaml(BUNDLE / ROOT_FILE_NAME) or {}).keys())
    extra, absent = [k for k in ours if k not in SAMPLE["top_keys"]], [k for k in SAMPLE["top_keys"] if k not in ours]
    if extra or absent:
        callout(f"Schema differs from the sample root config. Ours adds <code>{esc(extra)}</code>, sample-only <code>{esc(absent)}</code>. "
                "Check HARNESS_README, or set <code>CFG['bundle_base']='sample'</code>.", "warn")
    else:
        callout("Root config uses the same top-level keys as the official sample.", "ok")

<a id="validate"></a>
# 🛡️ 11. Validator & packaging
A pre-flight check modelled on the official rules. The allowed extensions and size limit are **parsed from the README when present**, and the source of every limit is shown. Checks: root config, YAML parses, every `!include`/`config_path` resolves *inside* the bundle, single base model, alias matches the sample, verified tools only, file types, size, hidden files, adapter completeness.

In [ ]:
def readme_extensions():
    for line in README.splitlines():
        if re.search(r"allowed|permitted|extension", line, re.I):
            exts = set(re.findall(r"(?<![\w/])\.(yaml|yml|md|txt|py|json|safetensors|bin|toml|jinja|j2)\b", line, re.I))
            if len(exts) >= 3:
                return {"." + e.lower() for e in exts}, line.strip()
    return None, None

def readme_size_limit():
    for line in README.splitlines():
        if re.search(r"size|archive|limit|uncompressed", line, re.I):
            m = re.search(r"(\d+(?:\.\d+)?)\s*(GiB|GB|MiB|MB)\b", line)
            if m:
                mult = {"gib": 1024**3, "gb": 1024**3, "mib": 1024**2, "mb": 1024**2}[m.group(2).lower()]
                return int(float(m.group(1)) * mult), line.strip()
    return None, None

_exts, _ext_line = readme_extensions()
ALLOWED_EXTS = _exts or {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
_size, _size_line = readme_size_limit()
MAX_BYTES = _size or 3 * 1024**3

def walk_yaml(obj, fn, key=None):
    fn(key, obj)
    if isinstance(obj, dict):
        for k, v in obj.items():
            walk_yaml(v, fn, k)
    elif isinstance(obj, list):
        for v in obj:
            walk_yaml(v, fn, key)

def validate_bundle(root, verbose=True):
    root = Path(root).resolve()
    checks = []
    add = lambda status, name, detail="": checks.append({"status": status, "check": name, "detail": detail})

    roots = [n for n in ROOT_NAMES if (root / n).exists()]
    add("✅" if len(roots) == 1 else "❌", "exactly one root config", ", ".join(roots) or "none found")

    parsed, models, tools_used, bad_refs = {}, set(), set(), []
    for y in sorted(p for p in root.rglob("*") if p.suffix in (".yaml", ".yml")):
        try:
            parsed[y] = load_yaml(y)
        except Exception as e:
            add("❌", f"YAML parses: {y.relative_to(root)}", str(e)[:120])
    add("✅" if parsed else "❌", "YAML files parse", f"{len(parsed)} file(s)")

    for y, data in parsed.items():
        def visit(key, v, y=y):
            if isinstance(v, dict) and v.get("__tag__") == "!include":
                refs = [v["value"]] if isinstance(v["value"], str) else []
            elif key == "config_path" and isinstance(v, str):
                refs = [v]
            else:
                refs = []
            for ref in refs:
                target = (y.parent / ref).resolve()
                if not target.exists() or root not in target.parents:
                    bad_refs.append(f"{y.relative_to(root)} → {ref}")
            if key == "model" and isinstance(v, str):
                models.add(v)
            if key == "tools" and isinstance(v, list):
                for t in v:
                    if isinstance(t, str):
                        tools_used.add(t)
        walk_yaml(data, visit)
    add("✅" if not bad_refs else "❌", "all !include / config_path references resolve inside the bundle", "; ".join(bad_refs) or "ok")
    add("✅" if len(models) == 1 else "❌", "single base model", ", ".join(sorted(models)) or "no model key found")
    if SAMPLE["models"]:
        add("✅" if models <= SAMPLE["models"] else "⚠️", "model alias matches sample submission", f"sample: {sorted(SAMPLE['models'])}")
    unknown = sorted(tools_used - set(AVAILABLE_TOOLS))
    add("✅" if not unknown else "⚠️", "only verified tools referenced", ", ".join(unknown) or f"{len(tools_used)} tools")

    files = [p for p in root.rglob("*") if p.is_file()]
    bad_ext = [str(p.relative_to(root)) for p in files if p.suffix.lower() not in ALLOWED_EXTS]
    add("✅" if not bad_ext else "❌", "allowed file types", ", ".join(bad_ext) or
        f"{sorted(ALLOWED_EXTS)} ({'README' if _exts else 'default, unverified'})")
    hidden = [str(p.relative_to(root)) for p in files if any(part.startswith(".") or part == "__pycache__" for part in p.relative_to(root).parts)]
    add("✅" if not hidden else "⚠️", "no hidden / cache files", ", ".join(hidden) or "ok")
    total = sum(p.stat().st_size for p in files)
    add("✅" if total <= MAX_BYTES else "❌", "size limit", f"{human_bytes(total)} / {human_bytes(MAX_BYTES)} ({'README' if _size else 'default, unverified'})")
    for st in [p for p in files if p.suffix == ".safetensors"]:
        ok = (st.parent / "adapter_config.json").exists()
        add("✅" if ok else "⚠️", f"adapter config next to {st.relative_to(root)}", "" if ok else "adapter_config.json missing")
    prompt_tok = sum(est_tokens(p.read_text(encoding="utf-8", errors="ignore")) for p in files if p.suffix == ".md")
    add("✅" if prompt_tok < 0.15 * CONTEXT_LIMIT else "⚠️", "prompt size vs context", f"~{prompt_tok:,} tokens across .md files")

    report = pd.DataFrame(checks)
    passed = not (report["status"] == "❌").any()
    if verbose:
        colour = lambda v: {"✅": "background-color:#E6F4EA", "⚠️": "background-color:#FEF7E0", "❌": "background-color:#FCE8E6"}.get(v, "")
        display(report.style.hide(axis="index").map(colour, subset=["status"]).set_properties(**{"text-align": "left"}))
        callout("All blocking checks passed." if passed else "Blocking issues found. Fix them before submitting.", "ok" if passed else "err")
    return passed, report

VALID, VALIDATION = validate_bundle(BUNDLE)

In [ ]:
def package(bundle, zip_path):
    """Deterministic zip: sorted entries, fixed timestamps → identical bytes for identical bundles."""
    zip_path = Path(zip_path)
    zip_path.unlink(missing_ok=True)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(p for p in Path(bundle).rglob("*") if p.is_file()):
            info = zipfile.ZipInfo(str(p.relative_to(bundle)).replace(os.sep, "/"), date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            z.writestr(info, p.read_bytes())
    return zip_path

ZIP_PATH = package(BUNDLE, WORK / CFG["zip_name"])
SHA = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()

# round trip: unzip into a temp dir and validate again (catches path/packaging mistakes)
_rt = WORK / "_roundtrip"
shutil.rmtree(_rt, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(_rt)
    names = z.namelist()
RT_OK, _ = validate_bundle(_rt, verbose=False)
shutil.rmtree(_rt, ignore_errors=True)

card = {"created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "zip": ZIP_PATH.name, "sha256": SHA,
        "architecture": ARCH, "bundle_base": CFG["bundle_base"], "model_alias": MODEL_ALIAS, "tools": AVAILABLE_TOOLS,
        "context_limit": CONTEXT_LIMIT, "sampling": {k: CFG[k] for k in ["temperature", "top_p", "top_k", "max_output_tokens", "thinking_budget"]},
        "prompt_sha": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:12], "validation_passed": bool(VALID and RT_OK)}
(WORK / "experiment_card.json").write_text(json.dumps(card, indent=2))

kpis([("submission.zip", human_bytes(ZIP_PATH.stat().st_size)), ("Files", len(names)), ("sha256", SHA[:12] + "…"),
      ("Validator", "PASSED ✅" if VALID else "FAILED ❌"), ("Round-trip", "PASSED ✅" if RT_OK else "FAILED ❌")])
display(HTML(pre("\n".join(names))))
callout(f"<b>{esc(ZIP_PATH)}</b> is ready. <code>experiment_card.json</code> records the config and hash, so you can log every submission "
        "and know exactly which prompt produced which score.", "ok" if VALID and RT_OK else "warn")

<a id="playbook"></a>
# 🚀 12. Playbook: what to try next, ranked by expected value

| # | Idea | Why (evidence from this notebook) | Effort |
|:-:|:--|:--|:-:|
| 1 | **Prompt iteration with an experiment log** | Every run writes `experiment_card.json`. Change one thing at a time and keep the table below up to date. | 🟢 |
| 2 | **Grep-first localization rule** | The mention boost in §7 shows how much exact identifier search adds over fuzzy retrieval. | 🟢 |
| 3 | **Mandatory reproduce → compile → test loop** | §8 shows how edits can apply cleanly yet break the file. Many tasks add new tests (§4) that check the *described* behaviour. | 🟢 |
| 4 | **Analyzer sub-agent tuning** | §9: the coder's context fills fast. Tighten the analyzer's answer format, or give it graph tools only for vague issues. | 🟡 |
| 5 | **Repository-specific hints** | Hot files (§5) and per-repo test commands can go in a short appendix of the system prompt. Keep it small, since it costs context on every turn. | 🟡 |
| 6 | **Reviewer sub-agent** | A read-only agent that inspects `git diff` against the issue before `submit_patch`. It catches scratch files, test edits and partial fixes (see `check_patch`). | 🟡 |
| 7 | **Sampling & thinking budget sweep** | Lower temperature for editing, and a thinking budget that does not starve tool calls. Track the effect in the log. | 🟢 |
| 8 | **LoRA adapter** (if the rules allow) | Train on *trajectories* (tool calls + edits) from repositories **other** than the benchmark ones. Tuning on the 129 public tasks risks overfitting and may conflict with the rules, so check them first. | 🔴 |

#### 🧾 Experiment log template
| run | architecture | prompt sha | temp | thinking | LB score | notes |
|:--|:--|:--|:-:|:-:|:-:|:--|
| baseline | analyzer+coder | *(from card)* | 0.2 | 4096 | | |

#### 🔧 How to adapt this starter
- **Everything** is controlled from `CFG` (§1). Prompts are plain functions (`build_system_prompt`, `build_analyzer_prompt`), so edit them directly.
- New data format? Extend `to_graph`, `unpack_embeddings` or `read_py_files`. The rest of the notebook picks it up.
- Validation fails on schema? Set `CFG["bundle_base"] = "sample"` to build on top of the official sample.

---
<div style="text-align:center;padding:12px;color:#5F6368">
Thanks to the community notebooks that explored this dataset first. This starter was rebuilt from scratch with corrected diff statistics and runtime-verified harness facts.<br>
<b>If this notebook saved you time, an upvote helps others find it. Good luck on the leaderboard! 🍀</b>
</div>